# LSTM Trading Signal Model — Train on Kaggle/Colab (Free T4 GPU)

## Overview
Trains a BiLSTM with attention for directional trading signal prediction.

## Instructions
1. Upload this notebook to [kaggle.com/code](https://kaggle.com/code) or Google Colab
2. Enable T4 GPU (Settings → Accelerator → GPU T4 x2)
3. Run All → download `lstm_model.pt` and `lstm_scaler.pkl`
4. Copy to `backend/models_artifacts/lstm_spy_daily_v1/`

In [ ]:
# Install dependencies (uncomment for Kaggle/Colab)
# !pip install torch pandas numpy scikit-learn matplotlib --quiet

import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

In [ ]:
# GPU configuration and AMP setup
import torch

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

# Enable cudnn benchmark for faster convolution operations on GPU
torch.backends.cudnn.benchmark = True

# Initialize GradScaler for mixed-precision (AMP) training
# Reduces memory usage and speeds up training on Tensor Cores
scaler = torch.cuda.amp.GradScaler()

print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}")
print(f"GradScaler initialized: {scaler is not None}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Configuration
SEQ_LEN = 60       # 60 trading days lookback
HORIZON = 1        # 1-day ahead prediction
BATCH_SIZE = 256
EPOCHS = 100
LR = 0.001
PATIENCE = 10

FEATURE_COLS = [
    "ret_1d", "ret_5d", "ret_20d",
    "rsi_14", "macd", "macd_signal",
    "bb_width", "atr_14",
    "volume_ratio", "log_volume",
]

print(f"Features: {len(FEATURE_COLS)}")
print(f"Sequence length: {SEQ_LEN} days")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute technical features. All features use shift(1) to avoid lookahead."""
    feat = df.copy()
    close = feat["close"]
    volume = feat["volume"]

    # Returns
    feat["ret_1d"]  = close.pct_change(1)
    feat["ret_5d"]  = close.pct_change(5)
    feat["ret_20d"] = close.pct_change(20)

    # RSI(14)
    delta = close.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    feat["rsi_14"] = 100 - (100 / (1 + gain / (loss + 1e-8)))

    # MACD
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    feat["macd"]        = macd_line / close
    feat["macd_signal"] = macd_line.ewm(span=9, adjust=False).mean() / close

    # Bollinger Band width
    sma20 = close.rolling(20).mean()
    std20 = close.rolling(20).std()
    feat["bb_width"] = (2 * std20) / (sma20 + 1e-8)

    # ATR(14)
    high, low = feat["high"], feat["low"]
    tr = pd.concat([
        (high - low),
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs(),
    ], axis=1).max(axis=1)
    feat["atr_14"] = tr.rolling(14).mean() / close

    # Volume features
    feat["volume_ratio"] = volume / (volume.rolling(20).mean() + 1)
    feat["log_volume"]   = np.log1p(volume)

    # Target: 1 if next day return > 0 else 0
    # Use shift(-1) for target (lookahead OK for target, not features)
    feat["target"] = (close.shift(-HORIZON) > close).astype(int)

    # Shift all features by 1 to prevent lookahead bias
    for col in FEATURE_COLS:
        feat[col] = feat[col].shift(1)

    return feat.dropna()


def make_sequences(features: np.ndarray, targets: np.ndarray, seq_len: int):
    X, y = [], []
    for i in range(seq_len, len(features)):
        X.append(features[i - seq_len: i])
        y.append(targets[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


print("Feature engineering functions defined.")

In [ ]:
class BiLSTMWithAttention(nn.Module):
    """Bidirectional LSTM with additive attention for trading signal prediction."""

    def __init__(
        self,
        input_size: int,
        hidden_size: int = 128,
        num_layers: int = 2,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.attention = nn.Linear(hidden_size * 2, 1)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_size * 2, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, input_size)
        lstm_out, _ = self.lstm(x)  # (batch, seq_len, hidden*2)
        # Additive attention over time steps
        attn_scores = self.attention(lstm_out).squeeze(-1)  # (batch, seq_len)
        attn_weights = torch.softmax(attn_scores, dim=-1).unsqueeze(-1)  # (batch, seq_len, 1)
        context = (lstm_out * attn_weights).sum(dim=1)  # (batch, hidden*2)
        out = self.dropout(context)
        return self.fc(out).squeeze(-1)  # (batch,) logits


# Test architecture
model_test = BiLSTMWithAttention(input_size=len(FEATURE_COLS))
x_test = torch.randn(4, SEQ_LEN, len(FEATURE_COLS))
print(f"Output shape: {model_test(x_test).shape}")  # should be (4,)
n_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

In [ ]:
def train_model(
    X_train, y_train, X_val, y_val,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
):
    """Train BiLSTMWithAttention using AMP mixed-precision."""
    model = BiLSTMWithAttention(input_size=X_train.shape[-1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.BCEWithLogitsLoss()

    X_tr = torch.FloatTensor(X_train)
    y_tr = torch.FloatTensor(y_train)
    X_v  = torch.FloatTensor(X_val).to(DEVICE)
    y_v  = torch.FloatTensor(y_val).to(DEVICE)

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")
    best_state = None
    no_improve = 0
    n_train = len(X_tr)

    for epoch in range(epochs):
        # ── Training with AMP ─────────────────────────────────────────
        model.train()
        train_loss = 0.0
        indices = torch.randperm(n_train)

        for start in range(0, n_train, batch_size):
            batch_idx = indices[start: start + batch_size]
            xb = X_tr[batch_idx].to(DEVICE)
            yb = y_tr[batch_idx].to(DEVICE)

            optimizer.zero_grad()

            # Mixed-precision forward pass
            with torch.cuda.amp.autocast():
                preds = model(xb)
                loss  = criterion(preds, yb)

            # Scaled backward pass
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * len(xb)

        train_loss /= n_train
        scheduler.step()

        # ── Validation ────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                val_logits = model(X_v)
                val_loss   = criterion(val_logits, y_v).item()
            val_preds = (torch.sigmoid(val_logits) > 0.5).float()
            val_acc   = (val_preds == y_v).float().mean().item()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{epochs}  "
                  f"train_loss={train_loss:.4f}  "
                  f"val_loss={val_loss:.4f}  "
                  f"val_acc={val_acc:.4f}")

        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if best_state:
        model.load_state_dict(best_state)
    return model, history


print("Training function defined.")

In [ ]:
# ── Generate synthetic data for demonstration ─────────────────────────────
# Replace with real OHLCV data from Alpaca/Yahoo Finance for production use
print("Generating synthetic OHLCV data for demonstration...")
np.random.seed(42)
n = 2000
price = 400.0 * np.cumprod(1 + np.random.normal(0.0003, 0.012, n))
demo_df = pd.DataFrame(
    {
        "open":   price * (1 + np.random.normal(0, 0.002, n)),
        "high":   price * (1 + np.abs(np.random.normal(0, 0.005, n))),
        "low":    price * (1 - np.abs(np.random.normal(0, 0.005, n))),
        "close":  price,
        "volume": np.random.randint(1_000_000, 5_000_000, n).astype(float),
    },
    index=pd.date_range("2019-01-01", periods=n, freq="B"),
)

# Engineer features
feat_df = make_features(demo_df)
print(f"Feature matrix: {feat_df.shape}")
print(f"Target balance: {feat_df['target'].mean():.2%} positive")

# Scale and split
raw_X = feat_df[FEATURE_COLS].values.astype(np.float32)
raw_y = feat_df["target"].values.astype(np.float32)

split = int(len(raw_X) * 0.80)
val_split = int(split * 0.85)

scaler_feat = StandardScaler()
train_feat = scaler_feat.fit_transform(raw_X[:split])
test_feat  = scaler_feat.transform(raw_X[split:])

X_all, y_all = make_sequences(train_feat, raw_y[:split], SEQ_LEN)
X_tr, X_val_ = X_all[:val_split], X_all[val_split:]
y_tr, y_val_ = y_all[:val_split], y_all[val_split:]
X_test, y_test = make_sequences(test_feat, raw_y[split:], SEQ_LEN)

print(f"Train: {X_tr.shape}, Val: {X_val_.shape}, Test: {X_test.shape}")

In [ ]:
# ── Train the model ────────────────────────────────────────────────────────
print("Starting training with AMP mixed-precision...")
print("=" * 60)

model, history = train_model(X_tr, y_tr, X_val_, y_val_)

# ── Plot training curves ────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="Train", color="#f5a623")
ax1.plot(history["val_loss"],   label="Val",   color="#00c853")
ax1.set_title("Loss (BCEWithLogits)")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history["val_acc"], color="#2196f3", label="Val Accuracy")
ax2.axhline(y=0.5, color="red", linestyle="--", alpha=0.5, label="Random")
ax2.set_title("Validation Accuracy")
ax2.set_xlabel("Epoch")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png")

In [ ]:
# ── Evaluate on held-out test set ──────────────────────────────────────────
from sklearn.metrics import accuracy_score, classification_report

model.eval()
with torch.no_grad():
    X_test_t = torch.FloatTensor(X_test).to(DEVICE)
    with torch.cuda.amp.autocast():
        test_logits = model(X_test_t)
    test_probs = torch.sigmoid(test_logits).cpu().numpy()
    test_preds = (test_probs > 0.5).astype(int)

test_acc = accuracy_score(y_test.astype(int), test_preds)
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")
print()
print(classification_report(
    y_test.astype(int), test_preds,
    target_names=["Down", "Up"]
))

In [ ]:
# ── Save model + scaler ────────────────────────────────────────────────────
import pickle

MODEL_PATH  = "lstm_model.pt"
SCALER_PATH = "lstm_scaler.pkl"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_config": {
            "input_size":   len(FEATURE_COLS),
            "hidden_size":  128,
            "num_layers":   2,
            "dropout":      0.3,
        },
        "feature_cols":     FEATURE_COLS,
        "seq_len":          SEQ_LEN,
        "test_accuracy":    float(test_acc),
    },
    MODEL_PATH,
)

with open(SCALER_PATH, "wb") as f:
    pickle.dump(scaler_feat, f)

print(f"Model saved  → {MODEL_PATH}")
print(f"Scaler saved → {SCALER_PATH}")
print(f"Test accuracy: {test_acc*100:.1f}%")
print()
print("Next steps:")
print("  1. Download lstm_model.pt and lstm_scaler.pkl from the output panel")
print("  2. Copy to backend/models_artifacts/lstm_spy_daily_v1/")
print("  3. The ML strategy will load them via ml/inference.py")